## 1. Install Library

In [ ]:
!pip install -q lmdb fire nltk natsort torch torchvision easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 34.1 MB/s eta 0:00:00


## 2. Download Dataset

In [ ]:
!gdown 1OVHHf8L4Lz5TrwhoCzaZLbEvHO_Mpsag

Downloading...
From: https://drive.google.com/uc?id=1nh7do0hKimI774JEdJ1KZSCUQbERAzv-
To: /content/dataset_easyocr.zip
100% 20.0M/20.0M [00:00<00:00, 37.0MB/s]


Unzip dataset

In [ ]:
!unzip -q dataset_easyocr_old.zip -d data

## 3. Split Dataset (Stratified)

In [ ]:
import os
import shutil
import random
from pathlib import Path

# ================= K O N F I G U R A S I =================
# Folder sumber yang berisi semua gambar dan gt.txt
SOURCE_FOLDER = 'data/dataset_easyocr'

# Folder tujuan output
OUTPUT_BASE = 'clean_data'

# Rasio pembagian (0.8 artinya 80% Train, 20% Validation)
TRAIN_RATIO = 0.8
# =========================================================

def split_dataset():
    # 1. Siapkan path folder tujuan
    train_dir = os.path.join(OUTPUT_BASE, 'train')
    val_dir = os.path.join(OUTPUT_BASE, 'val')

    # Hapus folder output lama jika ada (biar bersih)
    if os.path.exists(OUTPUT_BASE):
        shutil.rmtree(OUTPUT_BASE)

    os.makedirs(train_dir)
    os.makedirs(val_dir)

    print(f"Folder output dibuat: {OUTPUT_BASE}")

    # 2. Baca file gt.txt utama
    gt_path = os.path.join(SOURCE_FOLDER, 'gt.txt')
    if not os.path.exists(gt_path):
        print(f"Error: File gt.txt tidak ditemukan di {SOURCE_FOLDER}")
        return

    with open(gt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Bersihkan baris kosong
    lines = [line.strip() for line in lines if line.strip()]

    # ---------------------------------------------------------
    # MODIFIKASI: Stratify Split Logic
    # ---------------------------------------------------------
    top_lines = []
    bottom_lines = []
    other_lines = []

    # Pisahkan data berdasarkan nama file
    for line in lines:
        parts = line.split('\t')
        filename = parts[0]

        if '_top' in filename:
            top_lines.append(line)
        elif '_bottom' in filename:
            bottom_lines.append(line)
        else:
            other_lines.append(line) # Jaga-jaga jika ada nama lain

    # Acak masing-masing kategori secara independen
    random.shuffle(top_lines)
    random.shuffle(bottom_lines)
    random.shuffle(other_lines)

    # Hitung split index untuk masing-masing
    split_top = int(len(top_lines) * TRAIN_RATIO)
    split_bottom = int(len(bottom_lines) * TRAIN_RATIO)
    split_other = int(len(other_lines) * TRAIN_RATIO)

    # Bagi data
    train_top = top_lines[:split_top]
    val_top = top_lines[split_top:]

    train_bottom = bottom_lines[:split_bottom]
    val_bottom = bottom_lines[split_bottom:]

    # (Opsional) Data yang tidak sesuai pola masuk ke train/val juga
    train_other = other_lines[:split_other]
    val_other = other_lines[split_other:]

    # Gabungkan kembali
    train_data = train_top + train_bottom + train_other
    val_data = val_top + val_bottom + val_other

    # Acak ulang hasil gabungan agar urutan top/bottom tidak berkelompok saat training
    random.shuffle(train_data)
    random.shuffle(val_data)

    print(f"Total Data: {len(lines)}")
    print(f"  - Top    : {len(top_lines)}")
    print(f"  - Bottom : {len(bottom_lines)}")
    print("-" * 30)
    print(f"Train Set : {len(train_data)} (Top: {len(train_top)}, Btm: {len(train_bottom)})")
    print(f"Val Set   : {len(val_data)} (Top: {len(val_top)}, Btm: {len(val_bottom)})")
    print("-" * 30)

    # 4. Fungsi pembantu untuk menyalin (Tidak berubah)
    def process_split(data_lines, target_folder):
        new_gt_lines = []
        copied_count = 0

        for line in data_lines:
            parts = line.split('\t')
            if len(parts) < 2: continue

            filename = parts[0]
            label = parts[1] # Tidak digunakan untuk copy file, tapi untuk cek format

            src_img = os.path.join(SOURCE_FOLDER, filename)
            dst_img = os.path.join(target_folder, filename)

            # Copy gambar jika ada
            if os.path.exists(src_img):
                shutil.copy(src_img, dst_img)
                new_gt_lines.append(line)
                copied_count += 1
            else:
                # Debugging: Cek apakah ekstensi case sensitive atau salah nama
                print(f"Warning: Gambar {filename} tidak ditemukan di source.")

        # Tulis gt.txt baru di folder target
        target_gt_path = os.path.join(target_folder, 'gt.txt')
        with open(target_gt_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(new_gt_lines))

        return copied_count

    # 5. Eksekusi Copy
    print("Sedang menyalin data Train...")
    process_split(train_data, train_dir)

    print("Sedang menyalin data Val...")
    process_split(val_data, val_dir)

    print("\n" + "="*30)
    print("STRATIFIED SPLIT SELESAI")
    print("="*30)
    print(f"Output folder: {OUTPUT_BASE}")
    print("Siap untuk dikonversi ke LMDB.")

if __name__ == '__main__':
    split_dataset()

Folder output dibuat: clean_data
Total Data: 3886
  - Top    : 1943
  - Bottom : 1943
------------------------------
Train Set : 3108 (Top: 1554, Btm: 1554)
Val Set   : 778 (Top: 389, Btm: 389)
------------------------------
Sedang menyalin data Train...
Sedang menyalin data Val...

STRATIFIED SPLIT SELESAI
Output folder: clean_data
Siap untuk dikonversi ke LMDB.


## 4. Data Pre-processing

### 4.1 Data Augmentation

In [ ]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm

# ================= KONFIGURASI =================
INPUT_FOLDER = 'clean_data/train' # Folder train Anda (yg sdh dicrop)
OUTPUT_FOLDER = 'clean_data/train_augment'     # Folder baru hasil augmentasi
GT_FILE = 'gt.txt'                            # Nama file label
# ===========================================

def apply_blur(img):
    # Simulasi kamera goyang/tidak fokus
    ksize = random.choice([3, 5, 7])
    return cv2.GaussianBlur(img, (ksize, ksize), 0)

def apply_noise_dark(img):
    # Simulasi low light & noise
    # Gelapkan gambar
    factor = random.uniform(0.5, 0.8)
    img = cv2.convertScaleAbs(img, alpha=factor, beta=0)

    # Tambah noise (salt & pepper)
    noise = np.random.normal(0, 15, img.shape).astype(np.uint8)
    img = cv2.add(img, noise)
    return img

def apply_rotation(img):
    # Simulasi miring (Rotation + Shearing)
    rows, cols = img.shape[:2]

    # Random Angle (-10 sampai 10 derajat)
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((cols/2, rows/2), angle, 1)

    # Isi border dengan warna rata-rata pinggir agar tidak hitam legam
    border_val = int(np.mean(img[0, :]))
    dst = cv2.warpAffine(img, M, (cols, rows), borderValue=border_val)
    return dst

def augment_data():
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    input_gt_path = os.path.join(INPUT_FOLDER, GT_FILE)
    output_gt_path = os.path.join(OUTPUT_FOLDER, GT_FILE)

    if not os.path.exists(input_gt_path):
        print("GT File tidak ditemukan!")
        return

    with open(input_gt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_gt_lines = []

    print(f"Memproses {len(lines)} data asli...")

    for line in tqdm(lines):
        parts = line.strip().split('\t')
        if len(parts) < 2: continue

        filename, label = parts[0], parts[1]
        src_path = os.path.join(INPUT_FOLDER, filename)

        if not os.path.exists(src_path): continue

        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE) # Baca sbg grayscale
        if img is None: continue

        # 1. Simpan gambar asli
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, filename), img)
        new_gt_lines.append(f"{filename}\t{label}")

        # 2. Generate Variasi 1: Blur
        aug_name_1 = filename.replace('.jpg', '_blur.jpg')
        img_blur = apply_blur(img)
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, aug_name_1), img_blur)
        new_gt_lines.append(f"{aug_name_1}\t{label}")

        # 3. Generate Variasi 2: Gelap/Noise
        aug_name_2 = filename.replace('.jpg', '_dark.jpg')
        img_dark = apply_noise_dark(img)
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, aug_name_2), img_dark)
        new_gt_lines.append(f"{aug_name_2}\t{label}")

        # 4. Generate Variasi 3: Miring
        aug_name_3 = filename.replace('.jpg', '_rot.jpg')
        img_rot = apply_rotation(img)
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, aug_name_3), img_rot)
        new_gt_lines.append(f"{aug_name_3}\t{label}")

    # Simpan GT baru
    with open(output_gt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_gt_lines))

    print("\nSelesai!")
    print(f"Total data baru: {len(new_gt_lines)} (4x lipat dari asli)")
    print(f"Silakan buat LMDB dari folder: {OUTPUT_FOLDER}")

if __name__ == '__main__':
    augment_data()

Memproses 3108 data asli...


100%|██████████| 3108/3108 [00:06<00:00, 504.87it/s]


Selesai!
Total data baru: 12432 (4x lipat dari asli)
Silakan buat LMDB dari folder: clean_data/train_augment


### 4.2 Convert to LMBD Format

LMDB adalah format standar easy ocr

#### Membuat Script `create_lmdb.py`

In [ ]:
# 1. Buat script create_lmdb.py
%%writefile create_lmdb.py
import fire
import os
import lmdb
import cv2
import numpy as np

def checkImageIsValid(imageBin):
    if imageBin is None: return False
    try:
        imageBuf = np.frombuffer(imageBin, dtype=np.uint8)
        img = cv2.imdecode(imageBuf, cv2.IMREAD_GRAYSCALE)
        imgH, imgW = img.shape[0], img.shape[1]
    except: return False
    if imgH * imgW == 0: return False
    return True

def writeCache(env, cache):
    with env.begin(write=True) as txn:
        for k, v in cache.items():
            txn.put(k, v)

def createDataset(inputPath, gtFile, outputPath, checkValid=True):
    os.makedirs(outputPath, exist_ok=True)
    env = lmdb.open(outputPath, map_size=1099511627776)
    cache = {}
    cnt = 1

    with open(gtFile, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split('\t')
        if len(parts) < 2: continue
        imagePath, label = parts[0], parts[1]
        imagePath = os.path.join(inputPath, imagePath)

        if not os.path.exists(imagePath):
            print(f'{imagePath} does not exist')
            continue

        with open(imagePath, 'rb') as f:
            imageBin = f.read()

        if checkValid and not checkImageIsValid(imageBin):
            print(f'{imagePath} is not a valid image')
            continue

        imageKey = 'image-%09d'.encode() % cnt
        labelKey = 'label-%09d'.encode() % cnt
        cache[imageKey] = imageBin
        cache[labelKey] = label.encode()

        if cnt % 1000 == 0:
            writeCache(env, cache)
            cache = {}
            print('Written %d / %d' % (cnt, len(lines)))
        cnt += 1

    cache['num-samples'.encode()] = str(cnt-1).encode()
    writeCache(env, cache)
    print(f'Created dataset with {cnt-1} samples')

if __name__ == '__main__':
    fire.Fire(createDataset)

Overwriting create_lmdb.py


#### Convert ke Format LMDB Menggunakan Script `create_lmdb.py`

In [ ]:
# Konversi Train Data
!python create_lmdb.py --inputPath /content/clean_data/train_augment/ --gtFile /content/clean_data/train_augment/gt.txt --outputPath /content/data_lmdb/train/

# Konversi Val Data
!python create_lmdb.py --inputPath /content/clean_data/val/ --gtFile /content/clean_data/val/gt.txt --outputPath /content/data_lmdb/val/

Written 1000 / 12432
Written 2000 / 12432
Written 3000 / 12432
Written 4000 / 12432
Written 5000 / 12432
Written 6000 / 12432
Written 7000 / 12432
Written 8000 / 12432
Written 9000 / 12432
Written 10000 / 12432
Written 11000 / 12432
Written 12000 / 12432
Created dataset with 12432 samples
Created dataset with 778 samples


## 5. Setup EasyOCR

### 5.1 Clone EasyOCR

In [ ]:
!git clone https://github.com/ClovaAI/deep-text-recognition-benchmark

Cloning into 'deep-text-recognition-benchmark'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 499 (delta 208), reused 200 (delta 200), pack-reused 274 (from 1)
Receiving objects: 100% (499/499), 3.05 MiB | 8.52 MiB/s, done.
Resolving deltas: 100% (308/308), done.


### 5.2 Perbaikan Error Fungsi `_accumulate`

In [ ]:
import os

# Lokasi file yang error
file_path = '/content/deep-text-recognition-benchmark/dataset.py'

# Baca isi file
with open(file_path, 'r') as f:
    content = f.read()

# Baris yang menyebabkan error
bad_line = 'from torch._utils import _accumulate'

# Penggantinya: Kita definisikan manual fungsi _accumulate
replacement_code = '''
# --- FIX BY USER: _accumulate manual replacement ---
def _accumulate(iterable, fn=lambda x, y: x + y):
    "Return running totals"
    # _accumulate([1,2,3,4,5]) --> 1 3 6 10 15
    # _accumulate([1,2,3,4,5], operator.mul) --> 1 2 6 24 120
    it = iter(iterable)
    try:
        total = next(it)
    except StopIteration:
        return
    yield total
    for element in it:
        total = fn(total, element)
        yield total
# ---------------------------------------------------
'''

# Lakukan replace jika baris error masih ada
if bad_line in content:
    content = content.replace(bad_line, replacement_code)

    # Tulis ulang file
    with open(file_path, 'w') as f:
        f.write(content)
    print("Berhasil memperbaiki dataset.py! Error '_accumulate' seharusnya hilang.")
else:
    print("File sepertinya sudah diperbaiki sebelumnya (string tidak ditemukan).")

Berhasil memperbaiki dataset.py! Error '_accumulate' seharusnya hilang.


## 6. Custom Architecture

In [ ]:
import os

# 1. Menambahkan Class CustomAttentionCNN ke feature_extraction.py
feat_ext_path = '/content/deep-text-recognition-benchmark/modules/feature_extraction.py'

custom_cnn_code = """
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CustomAttentionCNN(nn.Module):
    '''
    Arsitektur khusus: CNN + Squeeze-and-Excitation Attention
    Didesain untuk mempertahankan fitur angka dan menekan noise background.
    '''
    def __init__(self, input_channel, output_channel=512):
        super(CustomAttentionCNN, self).__init__()

        # Konfigurasi stride untuk mereduksi tinggi(H) menjadi 1, dan menjaga lebar(W)
        self.ConvNet = nn.Sequential(
            nn.Conv2d(input_channel, 64, 3, 1, 1), nn.ReLU(True),
            nn.MaxPool2d(2, 2), # H/2, W/2

            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            SEBlock(128), # Attention Block
            nn.MaxPool2d(2, 2), # H/4, W/4

            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            SEBlock(256), # Attention Block
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)), # H/8, W/4 (Lebar dipertahankan)

            nn.Conv2d(256, 512, 3, 1, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(True),
            SEBlock(512), # Attention Block
            nn.Conv2d(512, 512, 3, 1, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)), # H/16, W/4

            nn.Conv2d(512, output_channel, 2, 1, 0), nn.BatchNorm2d(output_channel), nn.ReLU(True)
            # Output H akan menjadi 1 jika input H adalah 32. Jika input 64, butuh satu pool lagi.
        )

        # Penyesuaian khusus jika menggunakan input imgH=64
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))

    def forward(self, input):
        conv = self.ConvNet(input)
        # Paksa Height menjadi 1 berapapun resolusi inputnya
        conv = self.adaptive_pool(conv)
        return conv
"""

with open(feat_ext_path, 'r') as f:
    content = f.read()

if "class CustomAttentionCNN" not in content:
    with open(feat_ext_path, 'a') as f:
        f.write("\n" + custom_cnn_code)
    print("✅ CustomAttentionCNN ditambahkan ke feature_extraction.py")
else:
    print("ℹ️ CustomAttentionCNN sudah ada.")

# 2. Mendaftarkan Arsitektur ke model.py
model_path = '/content/deep-text-recognition-benchmark/model.py'
with open(model_path, 'r') as f:
    model_content = f.read()

# Cari tempat inisiasi FeatureExtraction
target_import = "from modules.feature_extraction import VGG_FeatureExtractor, RCNN_FeatureExtractor, ResNet_FeatureExtractor"
replace_import = "from modules.feature_extraction import VGG_FeatureExtractor, RCNN_FeatureExtractor, ResNet_FeatureExtractor, CustomAttentionCNN"

target_if = "if opt.FeatureExtraction == 'VGG':"
replace_if = """if opt.FeatureExtraction == 'Custom':
            self.FeatureExtraction = CustomAttentionCNN(opt.input_channel, opt.output_channel)
        elif opt.FeatureExtraction == 'VGG':"""

if "CustomAttentionCNN" not in model_content:
    model_content = model_content.replace(target_import, replace_import)
    model_content = model_content.replace(target_if, replace_if)
    with open(model_path, 'w') as f:
        f.write(model_content)
    print("✅ Custom Extractor didaftarkan ke model.py")
else:
    print("ℹ️ Custom Extractor sudah didaftarkan.")

✅ CustomAttentionCNN ditambahkan ke feature_extraction.py
✅ Custom Extractor didaftarkan ke model.py


## 7. Training

Pindah directory

In [ ]:
%cd deep-text-recognition-benchmark

/content/deep-text-recognition-benchmark


Mulai training

In [ ]:

!python train.py \
--train_data /content/data_lmdb/train/ \
--valid_data /content/data_lmdb/val \
--select_data / \
--batch_ratio 1.0 \
--Transformation TPS \
--FeatureExtraction Custom \
--SequenceModeling BiLSTM \
--Prediction CTC \
--data_filtering_off \
--workers 0 \
--batch_size 32 \
--num_iter 1000 \
--valInterval 200 \
--output_channel 512 \
--hidden_size 256 \
--imgH 64 --imgW 200 \
--character "0123456789"

--------------------------------------------------------------------------------
dataset_root: /content/data_lmdb/train/
opt.select_data: ['/']
opt.batch_ratio: ['1.0']
--------------------------------------------------------------------------------
dataset_root:    /content/data_lmdb/train/	 dataset: /
sub-directory:	/.	 num samples: 5832
num total samples of /: 5832 x 1.0 (total_data_usage_ratio) = 5832
num samples of / per batch: 32 x 1.0 (batch_ratio) = 32
--------------------------------------------------------------------------------
Total_batch_size: 32 = 32
--------------------------------------------------------------------------------
dataset_root:    /content/data_lmdb/val	 dataset: /
sub-directory:	/.	 num samples: 366
--------------------------------------------------------------------------------
model input parameters 64 200 20 1 512 256 11 25 TPS Custom BiLSTM CTC
Skip Transformation.LocalizationNetwork.localization_fc2.weight as it is already initialized
Skip Transform